In [1]:
import os
from typing import Any, cast

import torch
import torch.nn as nn
from torchvision.models import resnet18
import genesis as gs
import numpy as np

from collections import defaultdict

import matplotlib.pyplot as plt
from tensordict.nn import TensorDictModule, TensorDictSequential
from tensordict.nn.distributions import NormalParamExtractor

from tensordict import TensorDict
from torchrl.data import Composite, Bounded

from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (
    Compose,
    DoubleToFloat,
    ObservationNorm,
    StepCounter,
    TransformedEnv,
    Resize,
)
from torchrl.envs import EnvBase
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm

d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\modules\mcts\scores.py:574: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  PUCT = functools.partial(PUCTScore, c=5)  # AlphaGo default value
d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\modules\mcts\scores.py:575: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB = functools.partial(UCBScore, c=math.sqrt(2))  # default from Auer et al. 2002
d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\modules\mcts\scores.py:576: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB1_TUNED = functools.partial(
d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\modules\mcts\scores.py:579: FutureWarning: functools.partia

In [2]:
class ResNetEncoder(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()

        self.backbone = resnet18(weights=None if not pretrained else "IMAGENET1K_V1", norm_layer=self.gn)

        self.backbone.maxpool = nn.Identity()

        self.features = nn.Sequential(*list(self.backbone.children())[:-1])
        # [B, 512, 1, 1]

        self.flatten = nn.Flatten()

    def gn(self, num_channels):
        return nn.GroupNorm(num_groups=32, num_channels=num_channels)

    def forward(self, x):
        # x: [B, 3, H, W]
        x = x / 255.0
        x = self.features(x)
        x = self.flatten(x)  # [B, 512]
        return x

In [3]:
gs.init(backend=gs.cpu)

[Genesis] [20:27:31] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [20:27:31] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [20:27:31] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [20:27:31] [INFO] Running on [12th Gen Intel(R) Core(TM) i9-12900HX] with backend gs.cpu. Device memory: 23.74 GB.
[Genesis] [20:27:31] [INFO] 🚀 Genesis initialized. 🔖 version: 0.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO


In [12]:
gs.destroy()

[Genesis] [18:15:06] [INFO] 💤 Exiting Genesis and caching compiled kernels...


In [4]:
YOUBOT_DESCRIPTION = r"D:\ros\src\youbot_description"

JOINTS = [
    "wheel_joint_fl",
    "wheel_joint_fr",
    "wheel_joint_bl",
    "wheel_joint_br",
    "arm_joint_1",
    "arm_joint_2",
    "arm_joint_3",
    "arm_joint_4",
    "arm_joint_5",
    "gripper_finger_joint_l",
    "gripper_finger_joint_r",
]

joint2id = {k: i for i, k in enumerate(JOINTS)}

LINKS = [
    "arm_link_5",
]

link2id = {k: i for i, k in enumerate(LINKS)}

BATCH_RENDERER = False

num_joints = len(JOINTS)

In [ ]:
class VecEnv(EnvBase):
    def __init__(self, n_envs: int):
        super().__init__(batch_size=torch.Size([n_envs]))
        self.n_envs = n_envs
        
        self.scene = gs.Scene(
            show_viewer=False,
            viewer_options=gs.options.ViewerOptions(max_FPS=120),
            vis_options=gs.options.VisOptions(env_separate_rigid=True),
        )
        self.plane = self.scene.add_entity(gs.morphs.Plane())

        self.bot = self.scene.add_entity(
            gs.morphs.URDF(
                file=os.path.join(YOUBOT_DESCRIPTION, "robots/youbot.urdf"),
                pos=(0, 0, 0.1),
            ),
        )

        self.cylinder = self.scene.add_entity(
            gs.morphs.Cylinder(pos=(0.5, 0, 0.1), height=0.2, radius=0.02)
        )

        self.bot_dofs_idx = [self.bot.get_joint(name).dofs_idx_local[0] for name in JOINTS]
        self.bot_links_idx = [self.bot.get_link(name).idx_local for name in LINKS]

        if BATCH_RENDERER:
            renderer_opts = gs.sensors.BatchRendererCameraOptions(
                res=(640, 480),
                pos=(0.1, 0.0, 0.05),  # Offset from link frame
                lookat=(0.2, 0.0, 0.0),  # Look direction
                entity_idx=self.bot.idx,  # Attach to robot
                link_idx_local=self.bot_links_idx[0],  # End-effector link
                lights=[
                    {
                        "pos": (2.0, 2.0, 5.0),
                        "color": (1.0, 1.0, 1.0),
                        "intensity": 1.0,
                        "directional": True,
                        "castshadow": True,
                    }
                ],
            )
        else:
            renderer_opts = gs.sensors.RasterizerCameraOptions(
                res=(640, 480),
                pos=(0.1, 0.0, 0.05),  # Offset from link frame
                lookat=(0.2, 0.0, 0.0),  # Look direction
                entity_idx=self.bot.idx,  # Attach to robot
                link_idx_local=self.bot_links_idx[0],  # End-effector link
                # use_rasterizer=True,
            )

        self.camera = self.scene.add_sensor(cast(Any, renderer_opts))
        self.scene.build(n_envs=self.n_envs, env_spacing=(1.0, 1.0))

        self.observation_spec = Composite(
            observation=Bounded(
                low=0, high=255, 
                shape=(self.n_envs, 3, 480, 640), 
                dtype=torch.uint8
            ),
            shape=(self.n_envs,)
        )

        self.action_spec = Bounded(
            low=-1.0, high=1.0, 
            shape=(self.n_envs, num_joints),
            dtype=torch.float32,
        )

        self.reward_spec = Bounded(
            low=-float("inf"), high=0.0, 
            shape=(self.n_envs, 1)
        )
        
        self.done_spec = Composite(
            done=Bounded(low=0, high=1, shape=(self.n_envs, 1), dtype=torch.bool),
            terminated=Bounded(low=0, high=1, shape=(self.n_envs, 1), dtype=torch.bool),
            shape=(self.n_envs,)
        )

    def _get_observation(self):
        images = self.camera.read()
        return torch.tensor(images.rgb, dtype=torch.uint8).permute(0, 3, 1, 2)
    
    def _compute_reward(self):
        bots_links_pos = self.bot.get_links_pos(self.bot_links_idx)
        cylinder_pos = self.cylinder.get_links_pos()[:, 0]
        hands_pos = bots_links_pos[:, link2id["arm_link_5"]]

        distance = torch.sum((hands_pos - cylinder_pos) ** 2, -1) ** 0.5
        reward = -distance.unsqueeze(-1)
        return reward

    def _reset(self, tensordict=None):
        obs = self._get_observation()
        done = torch.zeros((self.n_envs, 1), dtype=torch.bool)
        
        return TensorDict({
            "pixels": obs,
            "done": done,
            "terminated": done
        }, batch_size=self.batch_size)
    
    def _step(self, tensordict):
        action = tensordict["action"]
        
        self.bot.set_dofs_kv(
            kv=action.cpu().numpy(),
            dofs_idx_local=self.bot_dofs_idx,
        )
        self.scene.step()

        obs = self._get_observation()
        reward = self._compute_reward()
        done = torch.zeros((self.n_envs, 1), dtype=torch.bool)

        return TensorDict({
            "pixels": obs,
            "reward": reward,
            "done": done,
            "terminated": done
        }, batch_size=self.batch_size)

    def _set_seed(self, seed: int):
        self.np_random = np.random.default_rng(seed)
        self.torch_generator = torch.Generator().manual_seed(seed)


In [37]:
env = VecEnv(1)
transformed_env = TransformedEnv(
    env,
    Resize(480 // 2, 640 // 2)
)

[Genesis] [20:52:05] [INFO] Scene <567e9c3> created.
[Genesis] [20:52:05] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 0, uid: <301fb3d>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [20:52:05] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 1, uid: <15cbb9d>, morph: <gs.morphs.URDF(file='D:\ros\src\youbot_description\robots\youbot.urdf')>, material: <gs.materials.Rigid>.


[Genesis] [20:52:06] [WARNING] Falling back to legacy URDF parser. Default values of physics properties may be off:
Error: no decoder found for mesh file 'D:\ros/src/youbot_description/meshes/youbot_base/base_convex.dae' - Element name 'base_convex', id 0
[Genesis] [20:52:06] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 2, uid: <c33c581>, morph: <gs.morphs.Cylinder>, material: <gs.materials.Rigid>.
[Genesis] [20:52:06] [INFO] Building scene <567e9c3>...
[Genesis] [20:52:06] [WARNING] Link 'gripper_finger_link_l' has dubious inertia [ixx=1.000e-02,iyy=1.000e-02,izz=1.000e-02] compared to the estimate from geometry [ixx=1.797e-06,iyy=1.714e-06,izz=4.084e-07] given material density 1500.000.
[Genesis] [20:52:06] [WARNING] Link 'gripper_finger_link_r' has dubious inertia [ixx=1.000e-02,iyy=1.000e-02,izz=1.000e-02] compared to the estimate from geometry [ixx=1.797e-06,iyy=1.714e-06,izz=4.084e-07] given material density 1500.000.
[Genesis] [20:52:09] [WARNING] Filtered out geometry p

In [38]:
frames_per_batch = 4096
total_frames = 2_000_000

num_epochs = 5
sub_batch_size = 256

lr = 1e-4
gamma = 0.995
lmbda = 0.97

clip_epsilon = 0.1
entropy_eps = 1e-3

max_grad_norm = 0.5

action_dim = num_joints

In [39]:
encoder_net = ResNetEncoder()

actor_net = nn.Sequential(
    nn.Linear(512, 256),
    nn.Tanh(),
    nn.Linear(256, 256),
    nn.Tanh(),
    nn.Linear(256, 2 * action_dim),
)

critic_net = nn.Sequential(
    nn.Linear(512, 256),
    nn.Tanh(),
    nn.Linear(256, 256),
    nn.Tanh(),
    nn.Linear(256, 1),
)

feature_extractor = TensorDictModule(
    encoder_net, in_keys=["pixels"], out_keys=["hidden"]
)

actor_head = TensorDictModule(
    actor_net, in_keys=["hidden"], out_keys=["loc_scale"]
)

extractor = TensorDictModule(
    NormalParamExtractor(), in_keys=["loc_scale"], out_keys=["loc", "scale"]
)

value_head = TensorDictModule(
    critic_net, in_keys=["hidden"], out_keys=["state_value"]
)

actor_module = TensorDictSequential(feature_extractor, actor_head, extractor)
value_module = TensorDictSequential(feature_extractor, value_head)

policy_module = ProbabilisticActor(
    module=actor_module,
    in_keys=["loc", "scale"],
    spec=transformed_env.action_spec,
    distribution_class=TanhNormal,
    return_log_prob=True,
)


In [40]:
collector = SyncDataCollector(
    transformed_env,
    policy_module,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
    split_trajs=False,
)

replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=frames_per_batch),
    sampler=SamplerWithoutReplacement(),
)

advantage_module = GAE(
    gamma=gamma, lmbda=lmbda, value_network=value_module, average_gae=True
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=clip_epsilon,
    entropy_bonus=bool(entropy_eps),
    entropy_coeff=entropy_eps,
    critic_coeff=1.0,
    loss_critic_type="smooth_l1",
)

optim = torch.optim.Adam(loss_module.parameters(), lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim, total_frames // frames_per_batch, 0.0
)

d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\collectors\_base.py:1045: DeprecationWarning: SyncDataCollector has been deprecated and will be removed in v0.13. Please use Collector instead.
  warnings.warn(
d:\miniconda3\envs\rl-env\Lib\site-packages\torchrl\collectors\_single.py:911: UserWarning: total_frames (2000000) is not exactly divisible by frames_per_batch (4096). This means 2944 additional frames will be collected.To silence this message, set the environment variable RL_WARNINGS to False.
  warnings.warn(
d:\miniconda3\envs\rl-env\Lib\site-packages\torch\utils\_device.py:116: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)


In [41]:
logs = defaultdict(list)
pbar = tqdm(total=total_frames)

for i, tensordict_data in enumerate(collector):
    
    with torch.no_grad():
        advantage_module(tensordict_data)

    data_view = tensordict_data.reshape(-1)
    replay_buffer.extend(data_view.cpu())

    # === PPO inner loop ===
    for _ in range(num_epochs):
        for _ in range(frames_per_batch // sub_batch_size):
            subdata = replay_buffer.sample(sub_batch_size)

            loss_vals = loss_module(subdata)

            loss = (
                loss_vals["loss_objective"]
                + loss_vals["loss_critic"]
                + loss_vals["loss_entropy"]
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_grad_norm)

            optim.step()
            optim.zero_grad()

    # === logging ===
    logs["reward"].append(tensordict_data["next", "reward"].mean().item())
    pbar.update(tensordict_data.numel())

    # === evaluation ===
    if i % 10 == 0:
        with set_exploration_type(ExplorationType.DETERMINISTIC), torch.no_grad():
            eval_rollout = transformed_env.rollout(1000, policy_module)
            logs["eval_reward"].append(eval_rollout["next", "reward"].mean().item())
            del eval_rollout

    scheduler.step()


  0%|          | 0/2000000 [01:51<?, ?it/s]

[Genesis] [20:52:16] [INFO] Running at 13.37 FPS (13.37 FPS per env, 1 envs).


[Genesis] [20:52:16] [INFO] Running at 13.40 FPS (13.40 FPS per env, 1 envs).
[Genesis] [20:52:16] [INFO] Running at 13.43 FPS (13.43 FPS per env, 1 envs).
[Genesis] [20:52:16] [INFO] Running at 13.49 FPS (13.49 FPS per env, 1 envs).
[Genesis] [20:52:16] [INFO] Running at 13.56 FPS (13.56 FPS per env, 1 envs).
[Genesis] [20:52:16] [INFO] Running at 13.61 FPS (13.61 FPS per env, 1 envs).
[Genesis] [20:52:16] [INFO] Running at 13.60 FPS (13.60 FPS per env, 1 envs).
[Genesis] [20:52:16] [INFO] Running at 13.61 FPS (13.61 FPS per env, 1 envs).
[Genesis] [20:52:16] [INFO] Running at 13.64 FPS (13.64 FPS per env, 1 envs).
[Genesis] [20:52:16] [INFO] Running at 13.63 FPS (13.63 FPS per env, 1 envs).
[Genesis] [20:52:16] [INFO] Running at 13.62 FPS (13.62 FPS per env, 1 envs).
[Genesis] [20:52:17] [INFO] Running at 13.62 FPS (13.62 FPS per env, 1 envs).
[Genesis] [20:52:17] [INFO] Running at 13.60 FPS (13.60 FPS per env, 1 envs).
[Genesis] [20:52:17] [INFO] Running at 13.59 FPS (13.59 FPS per 

KeyboardInterrupt: 